In [1]:
import os
import pandas as pd

from met_council_wrangler import CubeTransit
from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_transit
from cube_wrangler import StandardTransit
from cube_wrangler import Parameters
from cube_wrangler import Project

from network_wrangler import Scenario
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler.transit import write_transit

import pickle

Geopandas is not using pyogrio as the I/O engine.                Install pyogrio to benefit from faster I/O.


In [2]:
%reload_ext autoreload
%autoreload 2

### I/O

In [3]:
input_transit_dir = r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin"
input_scenario_dir = r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks"

output_transit_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit")
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card", "transit_project_cards")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")

In [4]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

### Create Transit Project Card
Compare two transit.lin files and write out a yaml file with transit changes.

In [5]:
base_transit_source=os.path.join(input_transit_dir, "transit_rail.lin")
build_transit_source=os.path.join(input_transit_dir, "transit_rail_new.lin")
transit_shape_crosswalk_file=os.path.join(input_transit_dir,"line_name_xwalk.csv")

base_transit_network = CubeTransit.create_from_cube(
    transit_source = base_transit_source, 
    parameters = metcouncil_parameters,
    transit_shape_crosswalk_file = transit_shape_crosswalk_file,
    model_shape_id_column = "shp_index"
)

build_transit_network = CubeTransit.create_from_cube(
    transit_source = build_transit_source, 
    parameters = metcouncil_parameters,
    transit_shape_crosswalk_file = transit_shape_crosswalk_file, 
    model_shape_id_column = "shp_index"   
)

transit_project = Project.create_project(
    base_transit_network=base_transit_network,
    build_transit_network=build_transit_network,
    parameters=metcouncil_parameters,
)

transit_project.write_project_card(
    os.path.join(project_card_dir, "test.yml")
)

Creating a new Cube Transit instance
reading: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin\transit_rail.lin
Creating a new Cube Transit instance
reading: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin\transit_rail_new.lin


### User to fill in the metadata in the result transit project card yaml file, i.e., the `project` name in the card file

### Apply all project cards in a directory

In [6]:
link_file = os.path.join(input_scenario_dir, 'link.json')
node_file = os.path.join(input_scenario_dir, 'node.geojson')
shape_file = os.path.join(input_scenario_dir, 'shape.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [7]:
transit_net = load_transit(os.path.join(input_scenario_dir))

\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\transit\io.py:86: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping

In [8]:
base_scenario = {"road_net": roadway_net, "transit_net": transit_net}
version_0_scenario = create_scenario(base_scenario = base_scenario)

Base_scenario doesn't contain ['road_net', 'transit_net', 'applied_projects', 'conflicts']
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet s

In [9]:
version_01_scenario = create_scenario(
    base_scenario = version_0_scenario,
    project_card_filepath = project_card_dir,
)
version_01_scenario.transit_net.road_net = version_01_scenario.road_net

PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk 

In [10]:
version_01_scenario.apply_all_projects()

PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk 

In [11]:
version_01_scenario.applied_projects

['update transit', 'delete transit', 'add transit']

## Apply single project card
Can be used to apply single project card. Uncomment the four lines below to test it.

In [12]:
# from projectcard import read_card
# card_path = os.path.join(project_card_dir, "update_transit.yml")
# card = read_card(card_path)
# test = transit_net.apply(card, reference_road_net=roadway_net)

### Write out new transit network in both cube and standard formats

In [12]:
write_transit(version_01_scenario.transit_net, file_format="txt", out_dir= output_transit_dir, overwrite=True)

In [13]:
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_transit_dir, "line_name_xwalk.csv")
)    

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1253: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1265: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])


In [14]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_transit_dir, "transit.lin"))